## 연습 1: 접근 제어

당신은 전국 슈퍼마켓 체인인 **NorthMart Retail**의 데이터 엔지니어입니다. 당신의 팀은 고객 및 거래 데이터를 Unity Catalog에 저장합니다. 분석가가 쿼리를 실행하기 전에, 먼저 카탈로그와 스키마를 생성하고 고객 테이블을 채운 다음 — `retail-analysts` 그룹에 필요한 권한을 부여해야 합니다.

> ⚠️ 이 연습을 시작하기 전에, Databricks에서 `retail-analysts` 그룹을 생성하고 본인의 사용자 계정을 그 그룹에 추가했는지 확인하세요. 랩 설정 지침을 참고하세요.

### 작업 1.1 — 카탈로그, 스키마, customers 테이블 생성

아래 셀은 이 연습에 필요한 모든 객체를 설정합니다:
- 모든 NorthMart 데이터 자산을 담을 `retail_catalog` 카탈로그를 생성합니다.
- 그 안에 `security_lab` 스키마를 생성합니다.
- 관리형 Delta 테이블 `retail_catalog.security_lab.customers`를 생성하고 6개의 샘플 고객 레코드로 채웁니다. 이 행들은 이 연습의 나머지 부분 전체에서 행 필터링과 컬럼 마스킹을 시연하는 데 사용됩니다.

| customer_id | customer_name   | email                   | region | total_spend | tier     |
|-------------|-----------------|-------------------------|--------|-------------|----------|
| 1           | Alice Johnson   | alice@northmart.com     | North  | 5200.00     | Standard |
| 2           | Bob Martinez    | bob@northmart.com       | South  | 8100.00     | Premium  |
| 3           | Carol Lee       | carol@northmart.com     | North  | 3400.00     | Standard |
| 4           | David Chen      | david@northmart.com     | East   | 7800.00     | Premium  |
| 5           | Eve Patel       | eve@northmart.com       | West   | 6200.00     | Standard |
| 6           | Frank Kim       | frank@northmart.com     | South  | 4100.00     | Standard |

셀을 실행하여 이 객체들을 프로비저닝하세요.

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속받습니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_ws*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"저장소 루트: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS retail_catalog
    MANAGED LOCATION '{storage_root}'
    COMMENT 'NorthMart Retail 데이터 자산을 위한 카탈로그'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_catalog.security_lab
  COMMENT '보안 랩 연습에 사용되는 스키마';

CREATE OR REPLACE TABLE retail_catalog.security_lab.customers AS
SELECT * FROM VALUES
  (1, 'Alice Johnson', 'alice@northmart.com', 'North', 5200.00, 'Standard'),
  (2, 'Bob Martinez',  'bob@northmart.com',   'South', 8100.00, 'Premium'),
  (3, 'Carol Lee',     'carol@northmart.com', 'North', 3400.00, 'Standard'),
  (4, 'David Chen',    'david@northmart.com', 'East',  7800.00, 'Premium'),
  (5, 'Eve Patel',     'eve@northmart.com',   'West',  6200.00, 'Standard'),
  (6, 'Frank Kim',     'frank@northmart.com', 'South', 4100.00, 'Standard')
  AS t(customer_id, customer_name, email, region, total_spend, tier);

### 작업 1.2 — retail-analysts 그룹에 권한 부여

`retail-analysts` 그룹은 `customers` 테이블을 쿼리해야 합니다. Unity Catalog 계층의 필요한 모든 수준에서 필요한 권한을 부여하세요:

- `retail_catalog`에 대한 `USE CATALOG`
- `retail_catalog.security_lab`에 대한 `USE SCHEMA`
- `retail_catalog.security_lab.customers`에 대한 `SELECT`

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"Databricks 그룹이 Unity Catalog 테이블에서 SELECT할 수 있도록 하려면 어떤 SQL grant가 필요한가요?"*

**힌트:** `GRANT <privilege> ON <object type> <object name> TO \`retail-analysts\``를 사용하세요.

In [ ]:
%sql
-- TODO: 부여된 권한을 부여하세요(GRANT)

### 작업 1.3 — 부여된 권한 확인

권한이 올바르게 할당되었는지 확인하세요. `SHOW GRANTS`를 사용하여 `customers` 테이블의 모든 권한을 표시하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"Unity Catalog 테이블의 권한을 확인하기 위해 SHOW GRANTS를 어떻게 사용하나요?"*

**예상 결과:** `retail-analysts`가 `SELECT` 권한과 함께 표시되어야 합니다.

In [ ]:
%sql
-- TODO: retail_catalog.security_lab.customers의 모든 grant를 표시하세요

## 연습 2: 행 필터링

NorthMart Retail의 지역 분석가는 자신의 지역에 해당하는 고객 레코드만 볼 수 있어야 합니다. 테이블을 쿼리하는 사람에 따라 행을 제한하는 **행 필터 함수(row filter function)** 를 구현합니다.

여러 사용자 계정을 생성할 수 없으므로, 본인의 사용자 계정을 *제외한* 모든 사람에게는 `North` 지역 행만 표시하고, 본인 계정은 모든 행을 볼 수 있도록 하는 필터를 작성합니다. 이는 지역 분석가가 경험하게 될 상황을 시뮬레이션합니다.

### 작업 2.1 — 행 필터 함수 생성

`STRING` 타입의 `region` 매개변수를 받아 다음 경우에 `TRUE`를 반환하는 SQL 함수 `retail_catalog.security_lab.region_filter`를 생성하세요:
- `region`이 `'North'`이거나, **또는**
- 현재 사용자가 본인 계정인 경우(`your@email.com`을 실제 이메일로 교체하세요).

즉, 권한이 있는 사용자가 아니라면 `North` 지역 행만 표시됩니다.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"Databricks Unity Catalog에서 current_user()를 사용하여 접근을 제어하는 행 필터 함수를 어떻게 생성하나요?"*

**힌트:** `CREATE OR REPLACE FUNCTION ... RETURN <current_user()를 사용하는 불리언 표현식>`을 사용하세요.

In [ ]:
%sql
-- TODO: retail_catalog.security_lab에 region_filter 함수를 생성하세요
-- 'your@email.com'을 실제 Databricks 사용자 이메일로 교체하세요

### 작업 2.2 — customers 테이블에 행 필터 적용

`region_filter` 함수를 `customers` 테이블에 연결하여 해당 테이블에 대한 모든 쿼리에 자동으로 적용되도록 하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"Azure Databricks에서 Unity Catalog 테이블에 행 필터를 어떻게 설정하나요?"*

**힌트:** `ALTER TABLE ... SET ROW FILTER <function> ON (<column>)`을 사용하세요.

In [ ]:
%sql
-- TODO: region 컬럼에 대해 retail_catalog.security_lab.customers에 region_filter를 적용하세요

### 작업 2.3 — 행 필터 테스트

`customers` 테이블을 쿼리하세요. 필터 함수에서 본인의 이메일이 권한 있는 사용자로 지정되어 있으므로, **6개 행 전부** 를 볼 수 있어야 합니다.

만약 다른 사람(본인 이메일이 아닌)으로 로그인했다면, `North` 지역의 2개 행만 보였을 것입니다.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"SQL을 사용하여 Databricks에서 현재 어떤 사용자가 쿼리를 실행 중인지 어떻게 확인할 수 있나요?"*

In [ ]:
%sql
-- 현재 사용자를 확인합니다
SELECT current_user();

-- TODO: retail_catalog.security_lab.customers를 쿼리하고 결과를 관찰하세요

➡️ 팁: 앞서 생성한 행 필터 함수에서 이메일 주소를 변경한 후 다시 테스트해 보세요. customers 테이블을 쿼리할 때의 출력을 관찰하세요.

### 작업 2.4 — 행 필터 제거

이제 `customers` 테이블에서 행 필터를 제거하세요. 제거한 후, 테이블을 다시 쿼리하여 사용자 컨텍스트와 관계없이 **6개 행 전부** 가 반환되는지 확인하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"Unity Catalog 테이블에서 행 필터를 어떻게 제거하나요?"*

**힌트:** `ALTER TABLE ... DROP ROW FILTER`를 사용하세요.

In [ ]:
%sql
-- TODO: retail_catalog.security_lab.customers에서 행 필터를 제거하세요

-- TODO: 테이블을 쿼리하여 6개 행 전부가 보이는지 확인하세요

## 연습 3: 컬럼 마스킹

고객 이메일 주소는 PII이며 권한이 없는 사용자에게는 마스킹되어야 합니다. 대부분의 사용자에게 전체 이메일 주소를 숨기는 **컬럼 마스크 함수(column mask function)** 를 생성하여 `customers` 테이블의 `email` 컬럼에 적용합니다.

### 작업 3.1 — 컬럼 마스크 함수 생성

`STRING` 타입의 `email` 매개변수를 받아 다음을 수행하는 SQL 함수 `retail_catalog.security_lab.mask_email`을 생성하세요:
- 현재 사용자가 본인 계정이면 **전체 이메일** 을 반환합니다.
- 그 외 모든 사용자에게는 **마스킹된 값**(예: `'***@***.com'`)을 반환합니다.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"Databricks Unity Catalog에서 current_user()를 사용하여 실제 값 또는 마스킹된 값을 반환할지 결정하는 컬럼 마스킹 함수를 어떻게 생성하나요?"*

**힌트:** 함수 본문 안에서 `CASE WHEN current_user() = '...' THEN email ELSE '***@***.com' END` 패턴을 사용하세요.

In [ ]:
%sql
-- TODO: retail_catalog.security_lab에 mask_email 함수를 생성하세요
-- 'your@email.com'을 실제 Databricks 사용자 이메일로 교체하세요

### 작업 3.2 — email 컬럼에 컬럼 마스크 적용

`mask_email` 함수를 `customers` 테이블의 `email` 컬럼에 컬럼 마스크로 적용하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"SQL을 사용하여 Azure Databricks에서 Unity Catalog 테이블 컬럼에 컬럼 마스크를 어떻게 설정하나요?"*

**힌트:** `ALTER TABLE ... ALTER COLUMN email SET MASK <function>`을 사용하세요.

In [ ]:
%sql
-- TODO: retail_catalog.security_lab.customers의 email 컬럼에 mask_email을 적용하세요

### 작업 3.3 — 컬럼 마스크 테스트

`customers` 테이블을 쿼리하세요. 본인이 마스크 함수에 지정된 권한 있는 사용자이므로, **실제 이메일 주소** 를 볼 수 있어야 합니다.

다른 사용자가 이 테이블을 쿼리하면 `email` 컬럼에서 `***@***.com`을 보게 됩니다.

In [ ]:
%sql
-- TODO: retail_catalog.security_lab.customers를 쿼리하고 email 컬럼을 관찰하세요

### 작업 3.4 — 컬럼 마스크 제거

`email` 컬럼에서 컬럼 마스크를 제거한 후, 테이블을 다시 쿼리하여 이제 모든 사용자에게 이메일이 완전히 보이는지 확인하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"Unity Catalog 테이블 컬럼에서 컬럼 마스크를 어떻게 제거(drop)하나요?"*

**힌트:** `ALTER TABLE ... ALTER COLUMN email DROP MASK`를 사용하세요.

In [ ]:
%sql
-- TODO: email 컬럼에서 컬럼 마스크를 제거하세요

-- TODO: 테이블을 쿼리하여 이제 이메일 주소가 마스킹 해제되었는지 확인하세요

## 연습 4: Azure Key Vault 비밀(Secret)

NorthMart Retail의 로열티 플랫폼은 연결을 위해 API 키가 필요합니다. 이 키를 노트북 안에 평문으로 저장하는 것은 보안 위험입니다. 이 연습에서는 랩 설정 중에 생성한 Azure Key Vault 기반 비밀 범위(secret scope)에서 API 키를 안전하게 가져옵니다.

> ⚠️ 이 연습의 셀을 실행하기 전에, 랩 지침의 Key Vault 설정 단계(Key Vault 생성, 비밀 `loyalty-api-key` 추가, 비밀 범위 `retail-kv-scope` 생성)를 완료했는지 확인하세요.

### 작업 4.1 — 범위 내 사용 가능한 비밀 나열

`loyalty-api-key`에 접근할 수 있는지 확인하기 위해 `retail-kv-scope` 비밀 범위에서 사용 가능한 모든 비밀을 나열하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"dbutils를 사용하여 Databricks 비밀 범위의 비밀을 어떻게 나열하나요?"*

**힌트:** `dbutils.secrets.list("scope-name")`을 사용하세요. 이는 키 이름만 반환하며 — 비밀 값은 절대 반환하지 않습니다.

In [ ]:
# TODO: retail-kv-scope의 모든 비밀을 나열하고 출력을 표시하세요

### 작업 4.2 — Python에서 API 키를 안전하게 가져오기

`retail-kv-scope` 범위에서 `loyalty-api-key` 비밀을 가져와 변수에 할당하세요. 그런 다음 출력하세요.

Azure Databricks가 노트북 출력에서 값을 **자동으로 마스킹(redact)** 하는 방식을 관찰하세요 — 실제 키 대신 `[REDACTED]`가 표시됩니다.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"dbutils.secrets.get을 사용하여 Databricks 비밀 범위에서 비밀을 어떻게 가져오며, 출력이 마스킹되는 이유는 무엇인가요?"*

In [ ]:
# TODO: retail-kv-scope에서 loyalty-api-key를 가져오세요
# TODO: api_key 변수를 출력하고 출력이 [REDACTED]로 표시되는지 관찰하세요

### 작업 4.3 — 비밀을 사용하여 연결 문자열 시뮬레이션

API 키를 포함하는 연결 문자열을 구성하여, NorthMart의 로열티 플랫폼에 연결하는 데 어떻게 사용되는지 시뮬레이션하세요. 연결 문자열을 출력하고 비밀 값이 여전히 마스킹되는지 관찰하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"값을 노출하지 않고 연결 문자열을 위한 f-string 안에서 Databricks 비밀을 어떻게 안전하게 사용하나요?"*

In [ ]:
# TODO: api_key 변수를 사용하여 연결 문자열을 구성하세요
# 예시 형식: "https://loyalty.northmart.com/api?key=<api_key>"
# TODO: 연결 문자열을 출력하세요

### 작업 4.4 — SQL을 사용하여 비밀 가져오기

`secret()` SQL 함수를 사용하여 동일한 비밀을 가져오세요. 이는 SQL 기반 연결을 구성할 때 유용합니다.

SQL 결과에서도 출력이 마스킹된다는 점에 유의하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 질문하세요:
> *"Databricks SQL에서 secret() 함수를 사용하여 범위에서 비밀을 어떻게 가져오나요?"*

In [ ]:
%sql
-- TODO: secret() SQL 함수를 사용하여 retail-kv-scope에서 loyalty-api-key를 가져오세요

## 정리(Cleanup)

이 랩에서 생성한 모든 객체를 제거하려면 아래 셀을 실행하세요.

In [ ]:
%sql
-- 모든 랩 객체를 제거합니다
ALTER TABLE retail_catalog.security_lab.customers DROP ROW FILTER;
ALTER TABLE retail_catalog.security_lab.customers ALTER COLUMN email DROP MASK;
DROP TABLE IF EXISTS retail_catalog.security_lab.customers;
DROP FUNCTION IF EXISTS retail_catalog.security_lab.region_filter;
DROP FUNCTION IF EXISTS retail_catalog.security_lab.mask_email;
DROP SCHEMA IF EXISTS retail_catalog.security_lab;
DROP CATALOG IF EXISTS retail_catalog CASCADE;